In [75]:
import torch
import einops

B = 2
n_heads = 4
T = 6

pattern = torch.arange(B * n_heads * T * T, dtype=torch.float32).view(B, n_heads, T, T)
induction_stripe = pattern.diagonal(offset=-3, dim1=-2, dim2=-1)
induction_score = einops.reduce(induction_stripe, "batch head_index position -> head_index", "mean")
print(induction_stripe)
print(induction_stripe[:,0])
print(induction_stripe[:,0].mean())
print(induction_stripe.mean(dim=0).mean(dim=1))
print(f"induction_score:\n{induction_score}")

tensor([[[ 18.,  25.,  32.],
         [ 54.,  61.,  68.],
         [ 90.,  97., 104.],
         [126., 133., 140.]],

        [[162., 169., 176.],
         [198., 205., 212.],
         [234., 241., 248.],
         [270., 277., 284.]]])
tensor([[ 18.,  25.,  32.],
        [162., 169., 176.]])
tensor(97.)
tensor([ 97., 133., 169., 205.])
induction_score:
tensor([ 97., 133., 169., 205.])


In [93]:
seq_len = 2
nhead = 3
direct_attributions = torch.arange(seq_len).unsqueeze(dim=-1)
l1_attributions = torch.arange(seq_len * nhead).view(seq_len, nhead)
l2_attributions = torch.arange(seq_len * nhead).view(seq_len, nhead)

print(f"direct_attributions:\n{direct_attributions}")
print(f"l1_attributions:\n{l1_attributions}")
print(f"l2_attributions:\n{l2_attributions}")
torch.concat([direct_attributions, l1_attributions, l2_attributions], dim=-1)
# seq_len n_components

direct_attributions:
tensor([[0],
        [1]])
l1_attributions:
tensor([[0, 1, 2],
        [3, 4, 5]])
l2_attributions:
tensor([[0, 1, 2],
        [3, 4, 5]])


tensor([[0, 0, 1, 2, 0, 1, 2],
        [1, 3, 4, 5, 3, 4, 5]])

In [92]:
a = torch.arange(seq_len)
b = torch.arange(seq_len)
print(f"a:\n{a}")
print(f"b:\n{b}")
torch.concat([a, b], dim=-1)

a:
tensor([0, 1])
b:
tensor([0, 1])


tensor([0, 1, 0, 1])

In [97]:
seq_len = 3

prefix = torch.arange(1)
rep_tokens_half = torch.arange(seq_len)
rep_tokens = torch.cat([prefix, rep_tokens_half, rep_tokens_half], dim=0)
print(f"rep_tokens:\n{rep_tokens}")
rep_tokens[-(seq_len - 1):]
# weird, why we only take the last seq_len - 1 tokens?

rep_tokens:
tensor([0, 0, 1, 2, 0, 1, 2])


tensor([1, 2])

In [104]:
batch = 2
seq_K = 3
d_head = 4
seq = 3
v = torch.arange(batch * seq_K * d_head, dtype=torch.float32).view(batch, seq_K, d_head)  # shape [batch seq_K d_head]
v_repeated = einops.repeat(v, "b sK h -> b sQ sK h", sQ=seq)

print(f"v:\n{v}")
print(f"v_repeated:\n{v_repeated}")
print(f"v_repeated.mean(dim=0):\n{v_repeated.mean(dim=0)}")

v:
tensor([[[ 0.,  1.,  2.,  3.],
         [ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.]],

        [[12., 13., 14., 15.],
         [16., 17., 18., 19.],
         [20., 21., 22., 23.]]])
v_repeated:
tensor([[[[ 0.,  1.,  2.,  3.],
          [ 4.,  5.,  6.,  7.],
          [ 8.,  9., 10., 11.]],

         [[ 0.,  1.,  2.,  3.],
          [ 4.,  5.,  6.,  7.],
          [ 8.,  9., 10., 11.]],

         [[ 0.,  1.,  2.,  3.],
          [ 4.,  5.,  6.,  7.],
          [ 8.,  9., 10., 11.]]],


        [[[12., 13., 14., 15.],
          [16., 17., 18., 19.],
          [20., 21., 22., 23.]],

         [[12., 13., 14., 15.],
          [16., 17., 18., 19.],
          [20., 21., 22., 23.]],

         [[12., 13., 14., 15.],
          [16., 17., 18., 19.],
          [20., 21., 22., 23.]]]])
v_repeated.mean(dim=0):
tensor([[[ 6.,  7.,  8.,  9.],
         [10., 11., 12., 13.],
         [14., 15., 16., 17.]],

        [[ 6.,  7.,  8.,  9.],
         [10., 11., 12., 13.],
         [14., 15., 16.

In [ ]:
torch.tensor([[1, 2],
              [2, 4],
              [3, 6]], dtype=torch.float32).mean(0)

tensor([1., 2.])

In [193]:
b = 1
seq = 4
n_heads = 1
d_head = 2

pattern = torch.arange(start=2, end=(b * seq * seq)+2, dtype=torch.float32).view(b, seq, seq)  # shape [batch seq seq]

v = torch.arange(start=1, end=(b * seq * d_head) + 1, dtype=torch.float32).view(b, seq, d_head)  # shape [batch seq d_head]
v_repeated = einops.repeat(v, "b sK h -> b sQ sK h", sQ=seq)
v_ablated = torch.zeros_like(v_repeated)

seq_posns = [1]

for offset in seq_posns:
    seqQ_slice = torch.arange(offset, seq)
    v_ablated[:, seqQ_slice, seqQ_slice - offset] = v_repeated[:, seqQ_slice, seqQ_slice - offset]
    # seq = 4
    # when offset = 1
    # seqQ_slice = [1, 2, 3]
    # seqQ_slice - offset = [0, 1, 2]
    # it means do not ablate (1,0), (2,1), (3,2)

z = torch.zeros((b, seq, n_heads, d_head), dtype=torch.float32)
z[:, :, 0] = einops.einsum(v_ablated, pattern, "b sQ sK h, b sQ sK -> b sQ h")

print(f"v:\n{v}")
print(f"v_repeated:\n{v_repeated}")
print(f"seqQ_slice:\n{seqQ_slice}")
print(f"v_ablated:\n{v_ablated}")
print(f"v_ablated[:,:,:,0]:\n{v_ablated[:, :, :, 0]}")
print(f"v_ablated[:,:,:,1]:\n{v_ablated[:, :, :, 1]}")
print(f"z:\n{z}")

v:
tensor([[[1., 2.],
         [3., 4.],
         [5., 6.],
         [7., 8.]]])
v_repeated:
tensor([[[[1., 2.],
          [3., 4.],
          [5., 6.],
          [7., 8.]],

         [[1., 2.],
          [3., 4.],
          [5., 6.],
          [7., 8.]],

         [[1., 2.],
          [3., 4.],
          [5., 6.],
          [7., 8.]],

         [[1., 2.],
          [3., 4.],
          [5., 6.],
          [7., 8.]]]])
seqQ_slice:
tensor([1, 2, 3])
v_ablated:
tensor([[[[0., 0.],
          [0., 0.],
          [0., 0.],
          [0., 0.]],

         [[1., 2.],
          [0., 0.],
          [0., 0.],
          [0., 0.]],

         [[0., 0.],
          [3., 4.],
          [0., 0.],
          [0., 0.]],

         [[0., 0.],
          [0., 0.],
          [5., 6.],
          [0., 0.]]]])
v_ablated[:,:,:,0]:
tensor([[[0., 0., 0., 0.],
         [1., 0., 0., 0.],
         [0., 3., 0., 0.],
         [0., 0., 5., 0.]]])
v_ablated[:,:,:,1]:
tensor([[[0., 0., 0., 0.],
         [2., 0., 0., 0.],
    